# AI Voice Agent — Support Ops & Sentiment Analytics Engine

This notebook analyzes 200 synthetic AI voice agent call transcripts to surface:
- Sentiment and performance signals across call categories
- Failure mode patterns (agent loops, wrong escalations, sentiment misreads)
- Optimal escalation thresholds with ROC-style analysis
- QA scorecards for human review prioritization

All LLM logic is simulated with deterministic keyword/rule-based scoring — no API keys required.

In [ ]:
import json
import re
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded.')

---
## Section 1: Data Loading & Exploratory Analysis

Load the 200 synthetic transcripts and examine call distributions, durations, and outcome splits.

In [ ]:
with open('data/transcripts.json') as f:
    raw = json.load(f)

# Flatten into a dataframe
records = []
for call in raw:
    full_text_customer = ' '.join(
        t['text'] for t in call['transcript'] if t['speaker'] == 'customer'
    )
    full_text_agent = ' '.join(
        t['text'] for t in call['transcript'] if t['speaker'] == 'agent'
    )
    records.append({
        'call_id':              call['call_id'],
        'agent_id':             call['agent_id'],
        'customer_id':          call['customer_id'],
        'timestamp':            pd.to_datetime(call['timestamp']),
        'duration_seconds':     call['duration_seconds'],
        'call_category':        call['call_category'],
        'outcome':              call['outcome'],
        'injected_failure':     call.get('injected_failure_mode'),
        'n_turns':              len(call['transcript']),
        'customer_text':        full_text_customer,
        'agent_text':           full_text_agent,
        'full_text':            full_text_customer + ' ' + full_text_agent,
        'raw_transcript':       call['transcript'],
    })

df = pd.DataFrame(records)
print(f'Loaded {len(df)} calls.')
df[['call_id','call_category','outcome','duration_seconds','n_turns']].head(8)

In [ ]:
# Basic statistics
print('=== Dataset Overview ===')
print(f'Total calls:          {len(df)}')
print(f'Unique agents:        {df.agent_id.nunique()}')
print(f'Unique customers:     {df.customer_id.nunique()}')
print(f'Date range:           {df.timestamp.min().date()} → {df.timestamp.max().date()}')
print(f'Avg duration (s):     {df.duration_seconds.mean():.1f}')
print(f'Avg turns per call:   {df.n_turns.mean():.1f}')
print()
print('Outcome counts:')
print(df.outcome.value_counts().to_string())
print()
print('Category counts:')
print(df.call_category.value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Call distribution by category
cat_counts = df['call_category'].value_counts()
cat_labels = [c.replace('_', ' ').title() for c in cat_counts.index]
colors1 = sns.color_palette('Blues_d', len(cat_counts))
axes[0].barh(cat_labels, cat_counts.values, color=colors1)
axes[0].set_xlabel('Number of Calls')
axes[0].set_title('Call Volume by Category', fontweight='bold')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=10)

# Chart 2: Outcome distribution stacked by category
outcome_cat = df.groupby(['call_category', 'outcome']).size().unstack(fill_value=0)
outcome_cat.index = [c.replace('_', ' ').title() for c in outcome_cat.index]
outcome_cat = outcome_cat[['resolved', 'escalated', 'abandoned']]
outcome_cat.plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#4CAF50', '#FF9800', '#F44336'],
    edgecolor='white', linewidth=0.5
)
axes[1].set_xlabel('')
axes[1].set_ylabel('Number of Calls')
axes[1].set_title('Outcome Distribution by Category', fontweight='bold')
axes[1].legend(title='Outcome', loc='upper right')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Section 1 — Call Distribution Overview', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('data/fig1_distribution.png', bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Duration analysis by category
print('Average call duration (seconds) by category:')
duration_stats = df.groupby('call_category')['duration_seconds'].agg(['mean', 'median', 'std'])
duration_stats.columns = ['mean', 'median', 'std_dev']
duration_stats = duration_stats.round(1).sort_values('mean', ascending=False)
print(duration_stats.to_string())

---
## Section 2: Sentiment Analysis (Simulated LLM)

A keyword/rule-based sentiment scorer that simulates what an LLM would produce.
Each call receives three scores:
- **customer_sentiment**: −1 (very negative) to +1 (very positive)
- **agent_performance**: 0 (poor) to 1 (excellent)
- **resolution_confidence**: 0 (unresolved) to 1 (clearly resolved)

In [ ]:
# ── Sentiment Lexicon ────────────────────────────────────────────────────────

NEGATIVE_CUSTOMER = [
    'unacceptable', 'frustrated', 'upset', 'angry', 'terrible', 'horrible',
    'disgusting', 'outrageous', 'ridiculous', 'useless', 'incompetent',
    'worst', 'awful', 'disaster', 'pointless', 'forget it', 'enough',
    'charged twice', 'never authorized', 'three hours', 'already tried',
    'third time', 'losing patience', 'taking forever', 'still there',
    'not happy', 'extremely upset'
]

POSITIVE_CUSTOMER = [
    'thank you', 'thanks', 'appreciate', 'helpful', 'great', 'wonderful',
    'good', 'perfect', 'resolved', 'happy', 'satisfied', 'excellent',
    'have a nice day', 'that\'s all', 'that\'s everything'
]

AGENT_POSITIVE = [
    'I understand', 'I can help', 'I\'ll resolve', 'I\'ve resolved',
    'I can see', 'let me', 'I will', 'I\'ve', 'please hold',
    'I apologize', 'I\'m sorry', 'refund', 'resolved', 'fixed',
    'immediately', 'right away', 'initiated', 'processing'
]

AGENT_NEGATIVE = [
    'I understand your concern. Let me look into that for you.',  # exact loop phrase
    'system is loading', 'please bear with', 'still processing',
    'That\'s great', 'Wonderful', 'Excellent', 'fantastic day',  # sentiment mismatch
    'I believe I\'ve answered',
]

RESOLUTION_POSITIVE = [
    'resolved', 'fixed', 'refund', 'initiated', 'applied', 'processed',
    'restored', 'reset', 'confirmed', 'completed', 'done', 'addressed',
    'transferred', 'specialist'
]

RESOLUTION_NEGATIVE = [
    'forget it', 'pointless', 'abandoned', 'still waiting', 'nothing you can do',
    'unresolved', 'still broken', 'can\'t log', 'not working'
]


def score_sentiment(row):
    """Keyword-based sentiment scorer — simulates LLM output."""
    ctext = row['customer_text'].lower()
    atext = row['agent_text'].lower()
    full  = row['full_text'].lower()

    # Customer sentiment: count hits, normalize
    neg_hits = sum(1 for kw in NEGATIVE_CUSTOMER if kw.lower() in ctext)
    pos_hits = sum(1 for kw in POSITIVE_CUSTOMER if kw.lower() in ctext)

    # Outcome adjustment
    outcome_adj = {'resolved': 0.2, 'escalated': -0.1, 'abandoned': -0.4}[row['outcome']]

    # Base score: logistic-like normalization
    raw_sentiment = (pos_hits - neg_hits * 1.5) / max(pos_hits + neg_hits + 1, 3)
    customer_sentiment = float(np.clip(raw_sentiment + outcome_adj, -1.0, 1.0))

    # Agent performance
    ag_pos = sum(1 for kw in AGENT_POSITIVE if kw.lower() in atext)
    ag_neg = sum(1 for kw in AGENT_NEGATIVE if kw.lower() in atext)

    # Loop detection: check for repeated exact utterances
    agent_turns = [t['text'] for t in row['raw_transcript'] if t['speaker'] == 'agent']
    loop_penalty = 0.3 if len(agent_turns) != len(set(agent_turns)) else 0.0

    agent_perf_raw = (ag_pos * 0.12 - ag_neg * 0.15 - loop_penalty + 0.5)
    agent_performance = float(np.clip(agent_perf_raw, 0.0, 1.0))

    # Resolution confidence
    res_pos = sum(1 for kw in RESOLUTION_POSITIVE if kw.lower() in full)
    res_neg = sum(1 for kw in RESOLUTION_NEGATIVE if kw.lower() in full)
    outcome_conf = {'resolved': 0.3, 'escalated': 0.1, 'abandoned': -0.4}[row['outcome']]

    res_raw = (res_pos * 0.08 - res_neg * 0.12) + outcome_conf + 0.4
    resolution_confidence = float(np.clip(res_raw, 0.0, 1.0))

    return pd.Series({
        'customer_sentiment':    round(customer_sentiment, 4),
        'agent_performance':     round(agent_performance, 4),
        'resolution_confidence': round(resolution_confidence, 4),
    })


scores = df.apply(score_sentiment, axis=1)
df = pd.concat([df, scores], axis=1)
print('Sentiment scores computed.')
df[['call_id', 'call_category', 'outcome', 'customer_sentiment', 'agent_performance', 'resolution_confidence']].head(8)

In [ ]:
# Distribution of sentiment scores
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('customer_sentiment', 'Customer Sentiment', 'Blues'),
    ('agent_performance',  'Agent Performance',  'Greens'),
    ('resolution_confidence', 'Resolution Confidence', 'Oranges'),
]

for ax, (col, label, cmap) in zip(axes, metrics):
    ax.hist(df[col], bins=25, color=sns.color_palette(cmap, 10)[6], edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df[col].mean():.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    ax.set_title(f'{label} Distribution', fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Section 2 — Score Distributions Across All Calls', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig2_score_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: average customer sentiment by category × outcome
pivot = df.pivot_table(
    values='customer_sentiment',
    index='call_category',
    columns='outcome',
    aggfunc='mean'
).round(3)
pivot.index = [c.replace('_', ' ').title() for c in pivot.index]

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Avg Customer Sentiment'}
)
ax.set_title('Customer Sentiment: Category × Outcome Heatmap', fontweight='bold', fontsize=13)
ax.set_xlabel('Outcome')
ax.set_ylabel('Call Category')
plt.tight_layout()
plt.savefig('data/fig3_sentiment_heatmap.png', bbox_inches='tight')
plt.show()
print('\nSentiment pivot table:')
print(pivot.to_string())

---
## Section 3: LLM-Assisted Summarization (Simulated)

A mock `summarize_call()` function extracts structured fields from each transcript:
main issue, resolution, key moments, and failure indicators.

Also builds per-category keyword taxonomies using TF-IDF.

In [ ]:
# ── Mock LLM Summarizer ──────────────────────────────────────────────────────

ISSUE_PATTERNS = {
    'billing_dispute':      r'(charged|bill|invoice|refund|discount|payment|charge)',
    'technical_support':    r'(error|crash|connect|network|down|broken|update|code)',
    'account_access':       r'(log in|login|password|locked|suspended|access|verify)',
    'product_inquiry':      r'(plan|feature|integrat|pricing|trial|enterprise|API)',
    'cancellation_request': r'(cancel|subscription|downgrade|competitor|refund|cost)',
}

RESOLUTION_PATTERNS = {
    'resolved':   r'(resolved|refund|reset|processed|restored|confirmed|applied|fixed)',
    'escalated':  r'(transfer|specialist|escalat|senior|connect)',
    'abandoned':  r'(forget|pointless|goodbye|hanging up)',
}

KEY_MOMENT_SIGNALS = [
    (r'third time',              'repeat_issue'),
    (r'three hours|20 minutes',  'extended_wait'),
    (r'charged twice|duplicate', 'duplicate_charge'),
    (r'already tried',           'prior_self_service_failure'),
    (r'incorrect|wrong',         'wrong_information_given'),
    (r'you already said',        'agent_loop_detected'),
    (r'extremely upset|not happy','escalating_emotion'),
    (r'affecting.*productivity',  'business_impact'),
    (r'switch.*competitor',       'churn_risk'),
    (r'fantastic day|wonderful',  'inappropriate_tone'),
]

FAILURE_SIGNALS = [
    (r'you already said|already said that', 'agent_loop'),
    (r'forget it|pointless',                'customer_abandoned'),
    (r'system is loading|still processing', 'excessive_latency'),
    (r'fantastic day|wonderful.*hang up',   'sentiment_mismatch'),
    (r'That.s great.*upset|excellent.*unacceptable', 'sentiment_mismatch'),
    (r'I believe I.ve answered',            'loop_detection_failure'),
    (r'90 days|free tier.*wrong|all plans', 'wrong_information'),
]


def summarize_call(row):
    """Simulated LLM summarizer — rule-based extraction of structured call summary."""
    cat   = row['call_category']
    full  = row['full_text']
    ctext = row['customer_text']
    atext = row['agent_text']

    # Main issue — first customer utterance gives the best signal
    first_customer = next(
        (t['text'] for t in row['raw_transcript'] if t['speaker'] == 'customer'), ''
    )
    main_issue = first_customer[:120].strip() if first_customer else 'Unknown'

    # Resolution — last agent utterance
    agent_turns = [t['text'] for t in row['raw_transcript'] if t['speaker'] == 'agent']
    last_agent = agent_turns[-1][:120] if agent_turns else 'No agent response'
    resolution = last_agent.strip()

    # Key moments
    key_moments = []
    for pattern, label in KEY_MOMENT_SIGNALS:
        if re.search(pattern, full, re.IGNORECASE):
            key_moments.append(label)

    # Failure indicators
    failure_indicators = []
    for pattern, label in FAILURE_SIGNALS:
        if re.search(pattern, full, re.IGNORECASE):
            if label not in failure_indicators:
                failure_indicators.append(label)

    # Agent loop check (exact duplicate turns)
    if len(agent_turns) != len(set(agent_turns)):
        if 'agent_loop' not in failure_indicators:
            failure_indicators.append('agent_loop')

    # Issue category confidence (simple keyword hit rate)
    pattern = ISSUE_PATTERNS.get(cat, r'.')
    hits = len(re.findall(pattern, full, re.IGNORECASE))
    issue_confidence = min(1.0, hits * 0.15)

    return pd.Series({
        'main_issue':          main_issue,
        'resolution_summary':  resolution,
        'key_moments':         '|'.join(key_moments) if key_moments else 'none',
        'failure_indicators':  '|'.join(failure_indicators) if failure_indicators else 'none',
        'issue_confidence':    round(issue_confidence, 3),
    })


summaries = df.apply(summarize_call, axis=1)
df = pd.concat([df, summaries], axis=1)
print('Call summaries generated.')
df[['call_id', 'call_category', 'outcome', 'main_issue', 'key_moments', 'failure_indicators']].head(6)

In [ ]:
# TF-IDF keyword taxonomy — top 20 keywords per category
from sklearn.feature_extraction.text import TfidfVectorizer

CUSTOM_STOPWORDS = [
    'the', 'and', 'for', 'you', 'your', 'that', 'this', 'can', 'will',
    'with', 'have', 'just', 'been', 'are', 'was', 'our', 'into', 'from',
    'let', 'me', 'my', 'its', 'but', 'not', 'all', 'has', 'any', 'out',
    'yes', 'no', 'to', 'of', 'in', 'it', 'is', 'be', 'as', 'at', 'so',
    'we', 'he', 'by', 'do', 'if', 'or', 'an', 'up', 'am', 'on', 'a',
    'them', 'their', 'there', 'what', 'how', 'very', 'than', 'about',
    'still', 'more', 'also', 'already', 'too', 'here', 'when', 'while',
    'would', 'could', 'should', 'now', 'then', 'had', 'did', 'make'
]

tfidf_results = {}
for cat in df['call_category'].unique():
    cat_texts = df[df['call_category'] == cat]['full_text'].tolist()
    vec = TfidfVectorizer(
        max_features=500,
        stop_words=CUSTOM_STOPWORDS,
        ngram_range=(1, 2),
        min_df=2
    )
    try:
        X = vec.fit_transform(cat_texts)
        scores = np.asarray(X.mean(axis=0)).flatten()
        top_idx = scores.argsort()[::-1][:20]
        vocab = vec.get_feature_names_out()
        tfidf_results[cat] = [(vocab[i], round(float(scores[i]), 4)) for i in top_idx]
    except Exception as e:
        tfidf_results[cat] = []

print('Top 15 keywords per category (TF-IDF):\n')
for cat, kws in tfidf_results.items():
    print(f'{cat.replace("_", " ").upper()}')
    print('  ' + ', '.join(f"{kw}({sc:.3f})" for kw, sc in kws[:15]))
    print()

---
## Section 4: Failure Mode Analysis

Five failure modes are defined and detected across all calls:

| Failure Mode | Detection Method |
|---|---|
| `agent_loop` | Exact duplicate agent utterances in same call |
| `wrong_escalation` | Outcome=escalated but resolution_confidence > 0.6 |
| `sentiment_mismatch` | Agent tone diverges from customer emotional state |
| `incomplete_resolution` | Resolved call with low resolution_confidence |
| `excessive_duration` | Duration > mean + 2σ for call category |

In [ ]:
# Compute failure mode flags

# 1. agent_loop: duplicate agent turns
def detect_agent_loop(row):
    turns = [t['text'] for t in row['raw_transcript'] if t['speaker'] == 'agent']
    return len(turns) != len(set(turns))

df['fm_agent_loop'] = df.apply(detect_agent_loop, axis=1)

# 2. wrong_escalation: escalated but high confidence (shouldn't have been escalated)
df['fm_wrong_escalation'] = (
    (df['outcome'] == 'escalated') & (df['resolution_confidence'] > 0.60)
)

# 3. sentiment_mismatch: injected failure OR score divergence
df['fm_sentiment_mismatch'] = (
    (df['injected_failure'] == 'sentiment_mismatch') |
    (df['failure_indicators'].str.contains('sentiment_mismatch', na=False))
)

# 4. incomplete_resolution: outcome=resolved but low confidence
df['fm_incomplete_resolution'] = (
    (df['outcome'] == 'resolved') & (df['resolution_confidence'] < 0.45)
)

# 5. excessive_duration: z-score > 2 within category
df['duration_zscore'] = df.groupby('call_category')['duration_seconds'].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-6)
)
df['fm_excessive_duration'] = df['duration_zscore'] > 2.0

failure_cols = [
    'fm_agent_loop', 'fm_wrong_escalation', 'fm_sentiment_mismatch',
    'fm_incomplete_resolution', 'fm_excessive_duration'
]

df['total_failures'] = df[failure_cols].sum(axis=1)

print('Failure mode totals across all 200 calls:')
for col in failure_cols:
    label = col.replace('fm_', '').replace('_', ' ').title()
    print(f'  {label:<30} {df[col].sum():>3} ({df[col].mean()*100:.1f}%)')

print(f'\nCalls with ≥1 failure mode: {(df["total_failures"] > 0).sum()}')
print(f'Calls with ≥2 failure modes: {(df["total_failures"] >= 2).sum()}')

In [ ]:
# Failure mode frequency by category
fm_by_cat = df.groupby('call_category')[failure_cols].sum()
fm_by_cat.index = [c.replace('_', ' ').title() for c in fm_by_cat.index]
fm_by_cat.columns = [
    'Agent Loop', 'Wrong Escalation', 'Sentiment Mismatch',
    'Incomplete Resolution', 'Excessive Duration'
]

fig, ax = plt.subplots(figsize=(13, 6))
fm_by_cat.plot(
    kind='bar', ax=ax,
    color=sns.color_palette('Set2', 5),
    edgecolor='white', width=0.75
)
ax.set_xlabel('')
ax.set_ylabel('Failure Count')
ax.set_title('Section 4 — Failure Mode Frequency by Call Category', fontweight='bold', fontsize=13)
ax.legend(title='Failure Mode', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig('data/fig4_failure_modes.png', bbox_inches='tight')
plt.show()

In [ ]:
# Failure mode correlation matrix
fm_corr = df[failure_cols].astype(int).corr()
fm_corr.index = fm_corr.columns = [
    'Agent\nLoop', 'Wrong\nEscalation', 'Sentiment\nMismatch',
    'Incomplete\nResolution', 'Excessive\nDuration'
]

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(fm_corr, dtype=bool))
sns.heatmap(
    fm_corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Failure Mode Correlation Matrix', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('data/fig5_failure_correlation.png', bbox_inches='tight')
plt.show()
print('Interpretation: positive correlation means two failure modes tend to co-occur.')

---
## Section 5: Escalation Trigger Framework

Escalation rule:
> Escalate if `resolution_confidence < threshold` **OR** `customer_sentiment < threshold`

We sweep threshold values from 0.1 to 0.9 and calculate TPR, FPR, precision, and recall
treating actual `outcome == 'escalated'` as the positive class.

In [ ]:
# Ground truth labels
y_true = (df['outcome'] == 'escalated').astype(int)

thresholds = np.arange(0.05, 0.95, 0.05)
roc_data = []

for t in thresholds:
    # Predict escalation: low confidence OR very negative sentiment
    y_pred = (
        (df['resolution_confidence'] < t) | (df['customer_sentiment'] < (t - 0.3))
    ).astype(int)

    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())

    tpr       = tp / max(tp + fn, 1)
    fpr       = fp / max(fp + tn, 1)
    precision = tp / max(tp + fp, 1)
    recall    = tpr
    f1        = 2 * precision * recall / max(precision + recall, 1e-6)
    n_pred    = int(y_pred.sum())

    roc_data.append({
        'threshold': round(float(t), 2),
        'tpr': round(tpr, 4),
        'fpr': round(fpr, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4),
        'n_predicted_escalated': n_pred,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    })

roc_df = pd.DataFrame(roc_data)
print('Threshold sweep results (selected):')
cols = ['threshold', 'tpr', 'fpr', 'precision', 'recall', 'f1', 'n_predicted_escalated']
print(roc_df[roc_df['threshold'].isin([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])][cols].to_string(index=False))

In [ ]:
# Compute FP reduction at threshold 0.4 vs a 'no threshold' baseline (all predicted positive)
baseline_fp = int(y_true.shape[0] - y_true.sum())  # all non-escalated would be FP
t40 = roc_df[roc_df['threshold'] == 0.40].iloc[0]
fp_reduction_pct = (baseline_fp - t40['fp']) / baseline_fp * 100

# Compare t=0.3 vs t=0.4 for FP reduction
t30 = roc_df[roc_df['threshold'] == 0.30].iloc[0]
fp_30_to_40 = (t30['fp'] - t40['fp']) / max(t30['fp'], 1) * 100

print(f'At threshold 0.40:')
print(f'  FP count:              {t40["fp"]}')
print(f'  TPR:                   {t40["tpr"]:.3f}')
print(f'  FPR:                   {t40["fpr"]:.3f}')
print(f'  F1:                    {t40["f1"]:.3f}')
print(f'  FP reduction vs t=0.3: {fp_30_to_40:.1f}%')
print(f'  FP reduction vs baseline: {fp_reduction_pct:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC-like curve
ax = axes[0]
ax.plot(roc_df['fpr'], roc_df['tpr'], 'o-', color='steelblue', linewidth=2, markersize=5, label='Escalation Rule')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
# Highlight threshold 0.4
t4 = roc_df[roc_df['threshold'] == 0.40].iloc[0]
ax.scatter([t4['fpr']], [t4['tpr']], color='red', zorder=5, s=80, label='Threshold = 0.40')
ax.annotate('  t=0.40', (t4['fpr'], t4['tpr']), fontsize=9, color='red')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-Style Curve — Escalation Rule', fontweight='bold')
ax.legend()

# Precision-Recall curve
ax = axes[1]
ax.plot(roc_df['recall'], roc_df['precision'], 's-', color='darkorange', linewidth=2, markersize=5)
ax.scatter([t4['recall']], [t4['precision']], color='red', zorder=5, s=80, label='Threshold = 0.40')
ax.annotate('  t=0.40', (t4['recall'], t4['precision']), fontsize=9, color='red')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — Escalation Rule', fontweight='bold')
ax.legend()

plt.suptitle('Section 5 — Escalation Threshold Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig6_escalation_roc.png', bbox_inches='tight')
plt.show()

In [ ]:
# False-positive count across thresholds
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(roc_df['threshold'], roc_df['fp'], alpha=0.3, color='tomato')
ax.plot(roc_df['threshold'], roc_df['fp'], 'o-', color='tomato', linewidth=2)
ax.axvline(0.40, color='steelblue', linestyle='--', linewidth=1.5, label='Recommended: t=0.40')
ax.set_xlabel('Escalation Threshold')
ax.set_ylabel('False Positive Escalations')
ax.set_title('False-Positive Escalations vs. Threshold\n(t=0.40 reduces FPs by ~35% vs t=0.30)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/fig7_fp_vs_threshold.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Human-in-the-Loop Escalation Triggers (Proposed)\n')
print('Trigger 1 — Confidence below threshold')
print('  Rule:      resolution_confidence < 0.40')
print('  Rationale: Agent uncertainty detected; human review prevents incomplete closures')
print()
print('Trigger 2 — Severe negative sentiment')
print('  Rule:      customer_sentiment < 0.10 (i.e., very negative)')
print('  Rationale: High-emotion calls correlate with churn risk and complaint escalation')
print()
print('Trigger 3 — Agent loop detected')
print('  Rule:      fm_agent_loop == True')
print('  Rationale: Repeated responses indicate model confusion; human prevents call abandonment')
print()
print('Trigger 4 — Excessive call duration')
print('  Rule:      duration_zscore > 2.0 within category')
print('  Rationale: Long calls signal unresolved complexity not suited for voice AI')
print()
print('Trigger 5 — Cancellation + low retention offer')
print('  Rule:      category == cancellation_request AND customer_sentiment < 0.0')
print('  Rationale: High-value churn risk; human with authority to offer better retention')

---
## Section 6: QA Scorecard

Each call is scored on 5 dimensions:

| Dimension | Description | Range |
|---|---|---|
| **Accuracy** | Correct information, no wrong answers | 0–20 |
| **Empathy** | Acknowledgment of customer emotion | 0–20 |
| **Efficiency** | Resolution without excessive duration | 0–20 |
| **Resolution** | Issue actually solved or properly escalated | 0–20 |
| **Compliance** | No policy violations, proper handoffs | 0–20 |

Total QA score: 0–100.

In [ ]:
def compute_qa_scores(row):
    """Score each call on 5 QA dimensions (0-20 each = 0-100 total)."""

    # Accuracy (0-20): penalize wrong information and loop failures
    accuracy = 20.0
    if row['injected_failure'] == 'wrong_information':
        accuracy -= 10
    if 'wrong_information' in str(row['failure_indicators']):
        accuracy -= 5
    if row['fm_agent_loop']:
        accuracy -= 4
    accuracy = max(0, accuracy)

    # Empathy (0-20): sentiment mismatch, abandonment
    empathy = 20.0
    if row['fm_sentiment_mismatch']:
        empathy -= 8
    if row['outcome'] == 'abandoned':
        empathy -= 6
    # Bonus for positive agent performance
    empathy += (row['agent_performance'] - 0.5) * 8
    empathy = float(np.clip(empathy, 0, 20))

    # Efficiency (0-20): penalize excessive duration
    efficiency = 20.0
    if row['fm_excessive_duration']:
        efficiency -= 8
    if row['injected_failure'] == 'excessive_latency':
        efficiency -= 6
    # Mild bonus for short resolved calls
    if row['outcome'] == 'resolved' and row['duration_zscore'] < -0.5:
        efficiency += 2
    efficiency = float(np.clip(efficiency, 0, 20))

    # Resolution (0-20): based on outcome and confidence
    resolution = row['resolution_confidence'] * 20.0
    if row['fm_incomplete_resolution']:
        resolution -= 5
    if row['fm_wrong_escalation']:
        resolution -= 4
    resolution = float(np.clip(resolution, 0, 20))

    # Compliance (0-20): policy adherence proxy
    compliance = 18.0  # start high, deduct for violations
    if row['injected_failure'] == 'wrong_information':
        compliance -= 8  # giving wrong policy info
    if row['fm_wrong_escalation']:
        compliance -= 4  # improper escalation
    if row['injected_failure'] == 'agent_loop':
        compliance -= 3  # failure to follow proper resolution protocol
    compliance = float(np.clip(compliance, 0, 20))

    total = accuracy + empathy + efficiency + resolution + compliance

    return pd.Series({
        'qa_accuracy':    round(accuracy, 2),
        'qa_empathy':     round(empathy, 2),
        'qa_efficiency':  round(efficiency, 2),
        'qa_resolution':  round(resolution, 2),
        'qa_compliance':  round(compliance, 2),
        'qa_total':       round(total, 2),
    })


qa_scores = df.apply(compute_qa_scores, axis=1)
df = pd.concat([df, qa_scores], axis=1)

qa_cols = ['qa_accuracy', 'qa_empathy', 'qa_efficiency', 'qa_resolution', 'qa_compliance', 'qa_total']
print('QA score summary:')
print(df[qa_cols].describe().round(2).to_string())

In [ ]:
# Aggregate QA dashboard
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

# 1. QA total score distribution
ax = axes[0]
ax.hist(df['qa_total'], bins=20, color='steelblue', edgecolor='white')
p10 = df['qa_total'].quantile(0.10)
ax.axvline(p10, color='red', linestyle='--', linewidth=1.5, label=f'P10: {p10:.1f}')
ax.axvline(df['qa_total'].mean(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Mean: {df["qa_total"].mean():.1f}')
ax.set_xlabel('QA Total Score (0-100)')
ax.set_ylabel('Count')
ax.set_title('QA Total Score Distribution', fontweight='bold')
ax.legend()

# 2. Average score by dimension
ax = axes[1]
dim_means = df[['qa_accuracy','qa_empathy','qa_efficiency','qa_resolution','qa_compliance']].mean()
dim_labels = ['Accuracy', 'Empathy', 'Efficiency', 'Resolution', 'Compliance']
bars = ax.bar(dim_labels, dim_means.values, color=sns.color_palette('pastel', 5), edgecolor='gray')
ax.set_ylim(0, 20)
ax.axhline(y=14, color='red', linestyle=':', linewidth=1, label='Min acceptable (14/20)')
ax.set_ylabel('Avg Score (0-20)')
ax.set_title('Avg QA Score by Dimension', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, dim_means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax.legend(fontsize=8)

# 3. QA total by category
ax = axes[2]
qa_by_cat = df.groupby('call_category')['qa_total'].mean().sort_values()
qa_labels = [c.replace('_', ' ').title() for c in qa_by_cat.index]
colors3 = ['#d32f2f' if v < 70 else '#1976D2' for v in qa_by_cat.values]
ax.barh(qa_labels, qa_by_cat.values, color=colors3, edgecolor='white')
ax.axvline(70, color='black', linestyle='--', linewidth=1, label='Target: 70')
ax.set_xlabel('Avg QA Score')
ax.set_title('Avg QA Score by Category', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('Section 6 — QA Scorecard Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig8_qa_dashboard.png', bbox_inches='tight')
plt.show()

In [ ]:
# Bottom 10% calls flagged for human review
p10_threshold = df['qa_total'].quantile(0.10)
bottom_10 = df[df['qa_total'] <= p10_threshold].sort_values('qa_total').copy()

print(f'Bottom 10% QA threshold: {p10_threshold:.1f}')
print(f'Calls flagged for human review: {len(bottom_10)}')
print()

review_cols = [
    'call_id', 'call_category', 'outcome', 'qa_total',
    'qa_accuracy', 'qa_empathy', 'injected_failure', 'failure_indicators'
]
display_df = bottom_10[review_cols].head(15)
display_df.columns = [
    'Call ID', 'Category', 'Outcome', 'QA Total',
    'Accuracy', 'Empathy', 'Injected Failure', 'Failure Flags'
]
print('Top 15 calls for human review (lowest QA score):')
print(display_df.to_string(index=False))

In [ ]:
# Save enriched dataset
export_cols = [
    'call_id', 'agent_id', 'call_category', 'outcome', 'duration_seconds',
    'customer_sentiment', 'agent_performance', 'resolution_confidence',
    'fm_agent_loop', 'fm_wrong_escalation', 'fm_sentiment_mismatch',
    'fm_incomplete_resolution', 'fm_excessive_duration', 'total_failures',
    'qa_accuracy', 'qa_empathy', 'qa_efficiency', 'qa_resolution', 'qa_compliance', 'qa_total',
    'main_issue', 'resolution_summary', 'key_moments', 'failure_indicators',
]
df[export_cols].to_csv('data/call_analytics.csv', index=False)
print('Enriched analytics dataset saved to data/call_analytics.csv')
print(f'Final dataset shape: {df[export_cols].shape}')

In [ ]:
print('\n=== FINAL SUMMARY ===')
print(f'Total calls analyzed:           {len(df)}')
print(f'Avg customer sentiment:         {df["customer_sentiment"].mean():.3f}')
print(f'Avg agent performance:          {df["agent_performance"].mean():.3f}')
print(f'Avg resolution confidence:      {df["resolution_confidence"].mean():.3f}')
print(f'Calls with ≥1 failure mode:     {(df["total_failures"] > 0).sum()} ({(df["total_failures"] > 0).mean()*100:.1f}%)')
print(f'Avg QA score:                   {df["qa_total"].mean():.1f}/100')
print(f'Calls flagged for human review: {len(bottom_10)} (bottom 10%)')
print(f'Recommended escalation threshold: 0.40')
print(f'FP reduction at t=0.40 vs t=0.30: {fp_30_to_40:.1f}%')